# PathSnap: Exploratory Data Analysis & Dataset Inventory
### Distributed Map-Matching & Route Reconstruction
**UE24CS352A: Machine Learning Hackathon**

This notebook conducts a structured exploratory data analysis (EDA) across the PathSnap dataset partitions:
- **Train Partition:** 80 trips used for candidate extraction, feature engineering, and cross-validation.
- **Student-Testing Partition:** 10 trips used for development-stage model evaluation and route reconstruction.

### Objectives:
1. **Dataset Inventory:** Trip counts, GPS observation counts, file structures, and road network statistics.
2. **Data-Quality & Anomaly Analysis:** GPS sampling rate regularity (nominal 1.0 Hz), irregular time gaps, spatial extents, and coordinate conventions.
3. **Geographical Data Handling:** Conversion from raw `(longitude, latitude)` to Leaflet/Folium `[latitude, longitude]` and geodesic distance considerations.
4. **Interactive Geographical Visualizations:** Folium maps visualizing raw GPS observations, road network geometry, reference routes, and candidate matches.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

TRAIN_DIR = '../Student folder/train'
TEST_DIR = '../Student folder/student_test'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Dataset Inventory & Partitions Overview

In [ ]:
# Load precomputed inventory
df_inv = pd.read_csv('../figures/dataset_inventory.csv')

print(f"Total trips indexed: {len(df_inv)}")
print(f"Training trips: {len(df_inv[df_inv['partition'] == 'train'])}")
print(f"Student-test trips: {len(df_inv[df_inv['partition'] == 'student_test'])}")

display(df_inv.groupby('partition')[['n_gps', 'duration_s', 'total_dist_km', 'n_nodes', 'n_arcs', 'n_route_arcs']].agg(['mean', 'std', 'min', 'max']).round(2))

## 2. Student Testing Set Inspection (10 Evaluation Trips)

In [ ]:
df_test = df_inv[df_inv['partition'] == 'student_test'][['trip_id', 'n_gps', 'duration_s', 'total_dist_km', 'pct_1hz', 'n_gaps_gt_5s', 'max_gap_s', 'n_arcs', 'n_route_arcs']]
display(df_test)

### Data-Quality Findings & Observations:
1. **Sampling Regularity:** Approximately **97.4%** of consecutive GPS observations have $\Delta t = 1.0\text{ s}$, confirming a nominal 1 Hz acquisition rate.
2. **GPS Dropouts / Anomalies:** While trips like `00000000`, `00000007`, `00000009` are perfectly contiguous with zero dropouts, trip `00000065` exhibits **5 significant GPS gaps**, and trip `00000076` exhibits **39 gaps** exceeding 5 seconds (with a maximum gap of 210s). Route reconstruction must detect and handle these splits to avoid bridging non-existent paths.
3. **Coordinate Systems:** The raw files contain `(lon, lat)` in decimal degrees. Because 1 degree of longitude scales as $\cos(\text{lat})$, ordinary Euclidean distances in degrees distort physical geometry. Our feature extraction transforms coordinates using local projection or geodesic haversine formulas.
4. **Road Network Scale:** Network sizes vary significantly—from 299 nodes in suburban/rural runs to over 155,000 nodes and 307,000 arcs in complex urban networks.

## 3. Visualizations: Sampling Intervals, Velocities & Network Topology

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sampling interval distribution
axes[0].imshow(plt.imread('../figures/gps_sampling_distribution.png'))
axes[0].axis('off')
axes[0].set_title('GPS Sampling Frequency Profile', fontsize=12, fontweight='bold')

# Speed distribution
axes[1].imshow(plt.imread('../figures/speed_distribution.png'))
axes[1].axis('off')
axes[1].set_title('Vehicle Velocity Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Interactive Folium Geographical Map (Trip 00000007)
Visualizing raw GPS observations (blue), nearby road network arcs (grey), and ground truth reference route (green).

In [ ]:
from src.folium_generator import create_trip_map

# Generate map for trip 00000007
map_path = create_trip_map('00000007')
print(f"Folium interactive map ready at: {map_path}")